# Preprocessamento e Split de Modelagem

Este notebook prepara a base analítica para as etapas de Machine Learning.

A partir da tabela `workspace.gold.ml_alfabetizacao_aluno`, são definidos:

- target;
- features utilizadas na modelagem;
- variáveis de identificação e controle;
- estratégia de separação entre treino, validação e teste.

A avaliação de 2024 será mantida integralmente como teste temporal final.

Os registros de 2023 serão utilizados para desenvolvimento do modelo. A separação entre treino e validação será realizada no nível do município, evitando que alunos de um mesmo contexto municipal sejam distribuídos entre os dois conjuntos.

O tratamento de valores ausentes e demais transformações será realizado posteriormente dentro do pipeline de Machine Learning, utilizando apenas informações do conjunto de treino.

# 1. Configuração
Nesta etapa são importadas as bibliotecas utilizadas no notebook

## Bibliotecas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

## Carregamento Gold

In [0]:
tabela_gold = "workspace.gold.ml_alfabetizacao_aluno"

df_gold = spark.table(tabela_gold)

print(f"Registros: {df_gold.count():,}")
print(f"Colunas: {len(df_gold.columns)}")

df_gold.printSchema()

# 2. Definição das Variáveis de Modelagem

As colunas da base analítica são classificadas de acordo com sua função na modelagem.

O target é `alfabetizado`, representado por:

- `1`: aluno alfabetizado;
- `0`: aluno não alfabetizado.

Identificadores como aluno, escola e município são preservados para rastreabilidade e validações, mas não são utilizados diretamente como features.

O ano também não será utilizado como variável explicativa, pois define a própria estratégia de separação temporal.

O peso amostral é preservado para análises posteriores, mas não será utilizado como feature.

Para evitar redundância e reduzir a influência de distribuições altamente assimétricas, PIB e população em valores brutos não entram inicialmente no modelo. São utilizadas suas transformações logarítmicas.

A primeira versão utiliza características de rede de ensino, contexto socioeconômico, infraestrutura escolar e contexto econômico e demográfico municipal.

In [0]:
target = "alfabetizado"

features_categoricas = ["rede"]

features_numericas = [
    "inse_2021",
    "censo_pct_agua_potavel",
    "censo_pct_energia_rede_publica",
    "censo_pct_esgoto_rede_publica",
    "censo_pct_biblioteca_sala_leitura",
    "censo_pct_laboratorio_informatica",
    "censo_pct_internet_aprendizagem",
    "censo_docentes_por_100_alunos",
    "censo_turmas_por_100_alunos",
    "log_pib_per_capita",
    "log_populacao",
]

features_modelo = features_categoricas + features_numericas


variaveis_identificacao = [
    "ano",
    "id_aluno",
    "id_escola",
    "id_municipio"
]

variaveis_controle = [
    "peso_aluno",
    "rede_descricao",
    "pib",
    "populacao",
    "pib_per_capita"
]

# 3. Base de modelagem

In [0]:
colunas_modelagem = (
    variaveis_identificacao + ["peso_aluno"] + features_modelo + [target]
)

df_modelagem = df_gold.select(*colunas_modelagem)

display(df_modelagem.limit(20))

# 4. Estratégia de Separação dos Dados

A divisão dos dados considera explicitamente sua dimensão temporal.

Todos os registros de 2024 são reservados para o teste final e não serão utilizados durante desenvolvimento, seleção ou ajuste dos modelos.

Os dados de 2023 são utilizados para treino e validação.

Como diversas features possuem granularidade municipal ou municipal por rede, uma divisão aleatória no nível do aluno poderia colocar alunos pertencentes ao mesmo município simultaneamente nos conjuntos de treino e validação.

Para reduzir esse risco, os municípios de 2023 são separados de forma determinística entre treino e validação. Dessa forma, todos os alunos de um município permanecem no mesmo conjunto.

A divisão utilizada é aproximadamente:

- 80% dos municípios de 2023 para treino;
- 20% dos municípios de 2023 para validação;
- 2024 integralmente reservado para teste temporal.

In [0]:
df_2023 = df_modelagem.filter(F.col("ano") == 2023)
df_teste = df_modelagem.filter(F.col("ano") == 2024)

In [0]:
df_municipios_split = (
    df_2023.select("id_municipio")
    .distinct()
    .withColumn("bucket", F.pmod(F.xxhash64("id_municipio"), F.lit(100)))
    .withColumn("split", F.when(F.col("bucket") < 80, "treino").otherwise("validacao"))
    .select("id_municipio", "split")
)

In [0]:
df_2023_split = df_2023.join(df_municipios_split, on="id_municipio", how="inner")

df_treino = df_2023_split.filter(F.col("split") == "treino").drop("split")

df_validacao = df_2023_split.filter(F.col("split") == "validacao").drop("split")

### Validação

In [0]:
df_resumo_split = (
    df_treino.withColumn("split", F.lit("treino"))
    .unionByName(df_validacao.withColumn("split", F.lit("validacao")))
    .unionByName(df_teste.withColumn("split", F.lit("teste")))
)

display(
    df_resumo_split.groupBy("split")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
        F.round(F.avg(F.col("alfabetizado").cast("double")) * 100, 2).alias(
            "taxa_alfabetizacao"
        ),
    )
    .orderBy("split")
)

In [0]:
municipios_treino = df_treino.select("id_municipio").distinct()

municipios_validacao = df_validacao.select("id_municipio").distinct()

qtd_municipios_sobrepostos = municipios_treino.join(
    municipios_validacao, on="id_municipio", how="inner"
).count()

print(
    "Municípios presentes simultaneamente "
    f"em treino e validação: {qtd_municipios_sobrepostos}"
)

# 5. Valores ausentes por conjunto

In [0]:
def resumo_nulos(df, nome_split):
    total = df.count()

    resultado = df.agg(
        *[
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in features_modelo
        ]
    ).collect()[0]

    return [
        (
            nome_split,
            coluna,
            resultado[coluna],
            round(resultado[coluna] / total * 100, 3),
        )
        for coluna in features_modelo
    ]

In [0]:
nulos_split = (
    resumo_nulos(df_treino, "treino")
    + resumo_nulos(df_validacao, "validacao")
    + resumo_nulos(df_teste, "teste")
)

df_nulos_split = spark.createDataFrame(
    nulos_split, ["split", "variavel", "qtd_nulos", "pct_nulos"]
)

display(df_nulos_split.filter(F.col("qtd_nulos") > 0).orderBy("variavel", "split"))

# 6. Persistência da Estratégia de Split

A base analítica permanece armazenada na tabela
`workspace.gold.ml_alfabetizacao_aluno`.

Para evitar duplicação desnecessária dos registros individuais, os conjuntos de treino, validação e teste não são persistidos separadamente.

É armazenado apenas o mapa de atribuição dos municípios aos conjuntos de modelagem.

Para 2023, cada município é classificado como treino ou validação. Para 2024, todos os municípios são classificados como teste temporal.

Essa tabela permite reconstruir de forma reproduzível os conjuntos utilizados durante o desenvolvimento e avaliação dos modelos.

In [0]:
df_split_2023 = df_municipios_split.withColumn("ano", F.lit(2023)).select(
    "ano", "id_municipio", "split"
)

df_split_2024 = (
    df_teste.select("id_municipio")
    .distinct()
    .withColumn("ano", F.lit(2024))
    .withColumn("split", F.lit("teste"))
    .select("ano", "id_municipio", "split")
)

df_split_modelagem = df_split_2023.unionByName(df_split_2024)

In [0]:
(
    df_split_modelagem.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.ml_split_modelagem")
)

In [0]:
display(
    df_split_modelagem.groupBy("ano", "split")
    .agg(F.countDistinct("id_municipio").alias("qtd_municipios"))
    .orderBy("ano", "split")
)

# 7. Conclusão do Preprocessamento e Split

A base analítica foi separada em conjuntos de treino, validação e teste respeitando a dimensão temporal e a estrutura contextual dos dados.

Os registros de 2023 foram divididos entre treino e validação no nível do município, garantindo que nenhum município esteja presente simultaneamente nos dois conjuntos.

A distribuição do target permaneceu semelhante entre treino e validação, indicando que a estratégia de separação preservou adequadamente a composição da população.

Os dados de 2024 foram mantidos integralmente como teste temporal final e não serão utilizados durante o desenvolvimento ou seleção dos modelos.

Os valores ausentes identificados serão tratados dentro do pipeline de Machine Learning, com os parâmetros de imputação ajustados exclusivamente sobre o conjunto de treino.

A modelagem mantém o target no nível individual, enquanto as principais variáveis explicativas representam características contextuais do município e da rede de ensino. Dessa forma, as previsões devem ser interpretadas como estimativas da probabilidade de alfabetização a partir do contexto educacional, socioeconômico e territorial disponível.